# Nightly research 2026-11-14 — P-2026-11-14-1: would the Spring 2026 reports' ECI claims survive a switch to oriented HS4 ECI?

**Question (from `proposals/2026-11-12-all-questions.md` #1, "Port the reports to oriented HS4 ECI (dry run)", which also covers `proposals/2026-11-11.md` #2 and `proposals/2026-11-13.md` #2):** For the 7 Spring 2026 report countries, recompute every ECI claim the report template makes (direction of change, standing against peers, level against peers) with the lab's notebook ECI, the sign-oriented grouped ECI, and oriented HS4 ECI. Which claims change? Which version agrees with the Growth Lab's published ECI?

**Why tonight:** Q01–Q16 are all taken on `main`, so per PROTOCOL §1.4 this is the top idea from `proposals/`. Proposal 2026-11-13 #2 (an HS4 ubiquity floor u ≥ 3 as a safety guard) is folded in as one variant.

- **Date:** 2026-11-14 (America/New_York)
- **Data:** `data/cleaned/2019–2023clean.csv` (18 groups); `data/raw/2019–2023_mega.csv` (HS6 → HS4); `willbetts/nightly/external-data/raw/atlas_hs92_country_year.csv` (Growth Lab published ECI, already in the repo, used only as a benchmark) and `atlas_hs92_product_year_4.csv` (not used). No downloads.
- **Peers:** from the "peer economies (…)" sentence of each `reports/2026-spring/*.pdf` (same dict as the Q02 run).
- **Library:** `willbetts/nightly/lib/eci.py` and `lib/research.py` (unchanged).
- **Run time:** printed in the last cell.

## TL;DR

- **Moving the reports to oriented HS4 ECI makes their ECI sentences agree with the Growth Lab's published ECI far more often.** Over 2019 → 2022, across 21 claims (7 countries × direction / peer standing / level rank among peers), the claims agree with ATLAS **16/21 (0.762)** under HS4, against **7/21 (0.333)** under the current notebook method (`NB`) and **11/21 (0.524)** under the sign-fixed grouped ECI (`G`). The ordering holds in both sensitivity windows: HS4 scores 0.810 for 2019→2021 and 0.857 for 2019→2023, against 0.524 and 0.571 for NB (Sections D, G).
- **The switch would rewrite most of the Spring 2026 ECI paragraphs.** From NB to H4, the `[increased/decreased]` word changes for 3 of 7 countries (Canada, UAE, UK). The peer-standing call changes for 3 of 7 when based on absolute change, and 5 of 7 when based on the printed % change. The rank among peers changes for 6 of 7. NB gets the peer-group level rank right in **0 of 7** cases, because the 2022 end year falls in the sign-flipped regime (Q01).
- **Even under HS4, many direction calls are noise.** Only **4 of 7** directions are robust, meaning they have the same sign under H4, H4u3, the fixed panel and ATLAS: Germany, Poland, UAE and UK. Canada (−0.022) and Singapore (−0.038) moved less than the **median one-year move of 0.071 SD**. Vietnam's sign depends on the sample: −0.072 under H4 and +0.065 on the fixed 112-country panel, while ATLAS gives +0.074. Sample churn alone flips 3 of 7 directions (H4 → H4fix).
- **The printed "% change" should be dropped.** It divides by a start value near 0. Under HS4, a −0.236 SD move for the UAE prints as **−158%** and a −0.072 SD move for Vietnam prints as **−179%**. Under NB, the UK prints **−394%**.
- **Two cheap checks come back clean.** An HS4 ubiquity floor u ≥ 3 changes **0** claims and at most **1** rank among report countries and peers in any year, so it's a free guard but not needed. HS4 PCI aggregated to the lab's groups orders 17 industries similarly to the grouped PCI (Spearman **0.922**), so Table 1's "most complex industries" wording largely survives. Services can't be scored at HS4.

## Why this matters for the lab

This is the **implementation** link. Q01 showed the notebook's ECI sign flips in 2021–2023. Q04, Q09 and Q13 showed that HS4 ECI is much closer to published ECI than the 18-group ECI (Spearman ≈ 0.97 against ≈ 0.69–0.77 in Q13). Those are **level** results across ~120 countries. But a report doesn't print a cross-section. It prints a **change** for one country (2019 → end year), a comparison with **three peers**, and an "increased/decreased" word. Changes are much noisier than levels. So the practical question for the next semester is whether switching the reports to oriented HS4 ECI changes what the reports *say*, and whether the new wording is closer to what the Growth Lab's published ECI says.

## Method

**Pre-stated before computing anything:**

- **Window:** 2019 → 2022 is primary. The reports say "2019 to 2024", but the repo has no 2024 data (Q01 data request), and 2023 is suspect (D1). 2019 → 2021 and 2019 → 2023 are sensitivity windows.
- **Claims** (from `templates/Auto Report Template.md` §2.2 and the executive summary), for each report country *c* with peers *P(c)*:
  - **C1 direction:** `[increased/decreased]` = sign of ECI_end − ECI_start (notebook cell 61 uses `begin > end`).
  - **C2 peer standing:** is *c*'s ECI change above or below the mean change of its available peers? It's computed two ways: **absolute** change (SD units, primary) and the notebook's **% change** `(end − start) / |start| × 100` (cell 42, which is what the reports print).
  - **C3 level standing:** *c*'s rank (1 = highest ECI) among {*c*} ∪ *P(c)* in the end year.
- **Variants:**
  - `NB`: the notebook as-is (exact port, `eci.notebook_eci`, drop-any-zero, strict RCA, unsorted/unoriented eigenvector). **Baseline:** this is what the Spring 2026 method produces on repo data.
  - `G`: grouped, lab rules, eigenvalues sorted, sign oriented to diversity (`research.grouped_eci`).
  - `H4`: HS4 from `data/raw`, reporters ≥ $1B, chapters 98/99 dropped, RCA ≥ 1, oriented (`research.repo_hs` + `eci.eci_from_exports`). **This is the candidate replacement.**
  - `H4u3`: H4 with products of ubiquity < 3 dropped (proposal 2026-11-13 #2).
  - `H4fix`: H4 computed each year on a **fixed** country panel (countries in the HS4 sample in all of 2019–2023), so changes aren't driven by sample churn (D2/D3).
  - `ATLAS`: Growth Lab published ECI (HS92). **Benchmark, not truth.** It uses mirror-reconciled trade and a population filter.
- **Primary metric:** the share of the 21 discrete claims (7 countries × C1/C2-absolute/C3) on which each variant agrees with `ATLAS`, 2019 → 2022. **Secondary metrics:** the number of claims that change from `NB` to `H4`, and the number of C1 direction calls that are **robust**, meaning they have the same sign under H4, H4u3, H4fix and ATLAS.
- Every variant tried is reported below. Merges are on ISO3.

In [1]:
import sys, time, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 170); pd.set_option('display.max_columns', 40)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]
REPORT = ['CAN', 'DEU', 'POL', 'SGP', 'ARE', 'GBR', 'VNM']
PEERS = {  # from the 'peer economies (...)' sentence of each reports/2026-spring/*.pdf (as in the Q02 run)
    'CAN': ['USA', 'AUS', 'GBR'], 'DEU': ['FRA', 'GBR', 'ITA'], 'POL': ['DEU', 'CZE', 'HUN'],
    'SGP': ['KOR', 'JPN', 'DEU'], 'ARE': ['SAU', 'KWT', 'QAT'], 'GBR': ['DEU', 'FRA', 'ITA'],
    'VNM': ['SGP', 'MYS', 'PHL']}
FOCUS = sorted(set(REPORT) | {p for v in PEERS.values() for p in v})
print('lib:', eci.__file__); print('focus countries:', len(FOCUS), FOCUS)

lib: /home/user/eci-research/willbetts/nightly/runs/2026-11-14_P-2026-11-14-1-report-claims-hs4/../../lib/eci.py
focus countries: 20 ['ARE', 'AUS', 'CAN', 'CZE', 'DEU', 'FRA', 'GBR', 'HUN', 'ITA', 'JPN', 'KOR', 'KWT', 'MYS', 'PHL', 'POL', 'QAT', 'SAU', 'SGP', 'USA', 'VNM']


### A. Build the five ECI variants for 2019–2023, plus the Atlas benchmark

In [2]:
V = {k: {} for k in ['NB', 'G', 'H4', 'H4u3', 'H4fix']}
HS4X, HS4M, PCI_G = {}, {}, {}
for y in YEARS:
    g = eci.load_groups(y, notebook_names=False)                  # original row order -> faithful NB sign
    s = eci.notebook_eci(g)
    iso = eci.group_name_to_iso3(s.index)
    s.index = [iso[n] if iso[n] else ('EU28' if 'EU' in str(n) else str(n)) for n in s.index]
    V['NB'][y] = s
    V['G'][y] = R.grouped_eci(y)[0]
    Xg = eci.notebook_filter(R.lab_groups_iso3(y))
    PCI_G[y] = eci.eci_pci(eci.mcp(Xg, strict=True))[1]
    X = R.repo_hs(y, 4); HS4X[y] = X
    M = eci.mcp(X); HS4M[y] = M
    V['H4'][y] = eci.eci_pci(M)[0]
    Mu = M.loc[:, M.sum(axis=0) >= 3]; Mu = Mu.loc[Mu.sum(axis=1) > 0]
    V['H4u3'][y] = eci.eci_pci(Mu)[0]
panel = sorted(set.intersection(*[set(HS4X[y].index) for y in YEARS]))
for y in YEARS:
    V['H4fix'][y] = eci.eci_from_exports(HS4X[y].loc[panel])
atlas = pd.read_csv('../../external-data/raw/atlas_hs92_country_year.csv')
V['ATLAS'] = {y: atlas[atlas.year == y].set_index('country_iso3_code')['eci'].dropna() for y in YEARS}
n = pd.DataFrame({k: {y: len(V[k][y]) for y in YEARS} for k in V}).T
n['focus_missing_2022'] = [', '.join(c for c in FOCUS if c not in V[k][2022].index) or '-' for k in V]
print('fixed HS4 panel size:', len(panel)); n

fixed HS4 panel size: 112


,2019,2020,2021,2022,2023,focus_missing_2022
NB,121,121,106,121,123,SAU
G,120,120,105,120,122,SAU
H4,133,130,136,119,131,SAU
H4u3,133,130,136,119,131,SAU
H4fix,112,112,112,112,112,SAU
ATLAS,230,230,230,230,230,-


Takeaway: sample sizes differ by variant. The ones that matter for the reports are shown in the last column: which report countries or peers have **no ECI in 2022**.

In [3]:
sp = pd.DataFrame({k: {y: R.spearman(V[k][y], V['ATLAS'][y])[0] for y in YEARS} for k in V if k != 'ATLAS'}).T.round(3)
sp.columns = [f'rho_vs_atlas_{y}' for y in YEARS]; sp

,rho_vs_atlas_2019,rho_vs_atlas_2020,rho_vs_atlas_2021,rho_vs_atlas_2022,rho_vs_atlas_2023
NB,0.687,0.737,-0.771,-0.770,-0.718
G,0.669,0.718,0.758,0.766,0.728
H4,0.972,0.974,0.966,0.969,0.967
H4u3,0.972,0.973,0.966,0.969,0.967
H4fix,0.969,0.970,0.970,0.969,0.969


Takeaway: this is a sanity check on **levels** that reproduces Q13. `NB` is negative in 2021–2023 because of the sign flip, H4 variants are around 0.97, and the u ≥ 3 floor and the fixed panel barely move the level fit.

### B. ECI paths of the focus countries

In [4]:
paths = {k: pd.DataFrame({y: V[k][y].reindex(FOCUS) for y in YEARS}) for k in V}
show = pd.concat({k: paths[k].loc[REPORT] for k in ['NB', 'G', 'H4', 'ATLAS']}, axis=0).round(2)
show

2019  2020  2021  2022  2023
NB    CAN -0.85 -0.54  0.61  0.64  0.37
      DEU  1.99  2.03 -1.86 -1.59 -1.50
      POL  0.95  0.38 -0.29 -0.62 -0.64
      SGP  0.76  0.54 -1.13 -1.52 -1.10
      ARE -2.08 -1.93  2.23  1.99  2.23
      GBR  0.51  0.89 -0.89 -1.50  2.62
      VNM  0.74  0.68 -0.61 -0.51 -0.46
G     CAN -1.06 -0.56 -0.57 -0.49 -0.16
      DEU  2.03  2.04  1.84  1.64  1.47
      POL  0.60  0.47  0.31  0.37  0.65
      SGP  0.84  1.21  1.22  1.64  1.15
      ARE -2.05 -2.19 -2.25 -1.86 -2.36
      GBR  0.46  0.68  0.89  1.49 -2.72
      VNM  0.69  0.78  0.59  0.48  0.50
H4    CAN  0.46  0.39  0.47  0.44  0.43
      DEU  1.84  1.75  1.83  1.72  1.79
      POL  0.83  0.72  0.77  0.62  0.67
      SGP  2.11  2.10  2.10  2.07  2.21
      ARE  0.15  0.13  0.10 -0.09  0.16
      GBR  1.44  1.42  1.62  1.55  1.50
      VNM  0.04  0.11  0.21 -0.03  0.22
ATLAS CAN  0.92  0.89  0.86  0.87  0.83
      DEU  1.54  1.51  1.52  1.52  1.50
      POL  1.06  1.05  1.05  1.03  1.02
      SGP  1.68  1.67  1.67  1.65  1.68
      ARE  0.70  0.71  0.69  0.69  0.70
      GBR  1.31  1.29  1.38  1.40  1.38
      VNM  0.40  0.45  0.55  0.48  0.58

Takeaway: `NB` jumps sign between 2020 and 2021 for every report country. `G` and `H4` move smoothly, but their *levels* disagree a lot for some countries (e.g. the UAE and Singapore, see the table).

In [5]:
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5), sharex=True)
sty = {'NB': ('#999999', ':'), 'G': ('#d95f02', '--'), 'H4': ('#1b9e77', '-'), 'ATLAS': ('#222222', '-.')}
for ax, c in zip(axes.flat, REPORT):
    for k, (col, ls) in sty.items():
        ax.plot(YEARS, paths[k].loc[c], ls, color=col, marker='o', ms=3, label=k)
    ax.axvspan(2022.5, 2023.5, color='0.9'); ax.set_title(c); ax.axhline(0, color='0.8', lw=0.6)
axes.flat[-1].axis('off'); axes.flat[0].legend(fontsize=8)
fig.suptitle('ECI paths, 2019-2023, by variant (2023 shaded: suspect, D1)'); fig.tight_layout()
fig.savefig(FIG / 'eci_paths_report_countries.png', dpi=90); plt.show()

### C. Recompute the report claims (2019 → 2022 primary)

In [6]:
def claims(k, y0, y1):
    E0, E1 = V[k][y0], V[k][y1]
    rows = []
    for c in REPORT:
        if c not in E0 or c not in E1 or pd.isna(E0.get(c)) or pd.isna(E1.get(c)):
            rows.append(dict(variant=k, country=c)); continue
        d = E1[c] - E0[c]; pct = d / abs(E0[c]) * 100
        pe = [p for p in PEERS[c] if p in E0 and p in E1 and pd.notna(E0[p]) and pd.notna(E1[p])]
        pd_abs = np.mean([E1[p] - E0[p] for p in pe]) if pe else np.nan
        pd_pct = np.mean([(E1[p] - E0[p]) / abs(E0[p]) * 100 for p in pe]) if pe else np.nan
        grp = E1.reindex([c] + PEERS[c]).dropna()
        rows.append(dict(variant=k, country=c, start=E0[c], end=E1[c], change=d, pct_change=pct,
                         C1_dir='increased' if d >= 0 else 'decreased',
                         peers_used=len(pe),
                         C2_abs=('above' if d > pd_abs else 'below') if pe else None,
                         C2_pct=('above' if pct > pd_pct else 'below') if pe else None,
                         C3_rank=f"{int(grp.rank(ascending=False)[c])}/{len(grp)}"))
    return pd.DataFrame(rows)

VARS = ['NB', 'G', 'H4', 'H4u3', 'H4fix', 'ATLAS']
CL = {w: pd.concat([claims(k, *w) for k in VARS], ignore_index=True) for w in [(2019, 2022), (2019, 2021), (2019, 2023)]}
prim = CL[(2019, 2022)]
pd.concat([CL[w].assign(window=f'{w[0]}-{w[1]}') for w in CL]).round(4).to_csv('report_claims_by_variant.csv', index=False)
prim.round(3)

,variant,country,start,end,change,pct_change,C1_dir,peers_used,C2_abs,C2_pct,C3_rank
0,NB,CAN,-0.853,0.643,1.496,175.431,increased,3,above,above,2/4
1,NB,DEU,1.988,-1.593,-3.581,-180.116,decreased,3,below,above,4/4
2,NB,POL,0.951,-0.622,-1.573,-165.402,decreased,3,above,above,1/4
3,NB,SGP,0.764,-1.518,-2.282,-298.705,decreased,3,above,below,2/4
4,NB,ARE,-2.079,1.992,4.070,195.818,increased,1,above,below,2/3
5,NB,GBR,0.509,-1.498,-2.007,-394.410,decreased,3,above,below,3/4
6,NB,VNM,0.741,-0.507,-1.248,-168.483,decreased,2,above,above,2/4
7,G,CAN,-1.058,-0.487,0.571,53.972,increased,3,above,below,3/4
8,G,DEU,2.031,1.643,-0.388,-19.097,decreased,3,below,below,1/4
9,G,POL,0.598,0.368,-0.229,-38.359,decreased,3,below,below,4/4


Takeaway: this is the full claim table for 2019 → 2022. Each row is what that variant would print in §2.2 of the report.

In [7]:
def wide(df, col):
    return df.pivot(index='country', columns='variant', values=col).reindex(index=REPORT, columns=VARS)
for col in ['C1_dir', 'C2_abs', 'C2_pct', 'C3_rank']:
    print(f'--- {col} (2019->2022)'); print(wide(prim, col).to_string()); print()

--- C1_dir (2019->2022)
variant         NB          G         H4       H4u3      H4fix      ATLAS
country                                                                  
CAN      increased  increased  decreased  decreased  increased  decreased
DEU      decreased  decreased  decreased  decreased  decreased  decreased
POL      decreased  decreased  decreased  decreased  decreased  decreased
SGP      decreased  increased  decreased  decreased  increased  decreased
ARE      increased  increased  decreased  decreased  decreased  decreased
GBR      decreased  increased  increased  increased  increased  increased
VNM      decreased  decreased  decreased  decreased  increased  increased

--- C2_abs (2019->2022)
variant     NB      G     H4   H4u3  H4fix  ATLAS
country                                          
CAN      above  above  above  above  below  below
DEU      below  below  below  below  below  below
POL      above  below  below  below  below  below
SGP      above  above  above  above

Takeaway: read across a row to see whether a report sentence survives the switch. The H4, H4u3 and H4fix columns should agree with each other if the HS4 call is stable.

### D. Pre-stated metrics: agreement with ATLAS, NB → H4 changes, robust direction calls

In [8]:
def agree_table(df):
    out = {}
    for k in VARS[:-1]:
        a = 0; tot = 0; per = {}
        for col in ['C1_dir', 'C2_abs', 'C3_rank']:
            x, b = wide(df, col)[k], wide(df, col)['ATLAS']
            ok = x.notna() & b.notna()
            if col == 'C3_rank':   # compare the rank only (group size can differ if a peer is missing)
                m = (x[ok].str.split('/').str[0] == b[ok].str.split('/').str[0])
            else:
                m = (x[ok] == b[ok])
            per[col] = f'{int(m.sum())}/{int(ok.sum())}'; a += int(m.sum()); tot += int(ok.sum())
        out[k] = {**per, 'total': f'{a}/{tot}', 'share': a / tot if tot else np.nan}
    return pd.DataFrame(out).T
AG = {w: agree_table(CL[w]) for w in CL}
print('Agreement with ATLAS, 2019->2022 (primary):'); print(AG[(2019, 2022)].to_string()); print()
for w in [(2019, 2021), (2019, 2023)]:
    print(f'Agreement with ATLAS, {w[0]}->{w[1]} (sensitivity):'); print(AG[w].to_string()); print()

Agreement with ATLAS, 2019->2022 (primary):
      C1_dir C2_abs C3_rank  total     share
NB       3/7    4/7     0/7   7/21  0.333333
G        3/7    4/7     4/7  11/21   0.52381
H4       6/7    5/7     5/7  16/21  0.761905
H4u3     6/7    5/7     5/7  16/21  0.761905
H4fix    5/7    6/7     5/7  16/21  0.761905

Agreement with ATLAS, 2019->2021 (sensitivity):
      C1_dir C2_abs C3_rank  total     share
NB       3/7    6/7     2/7  11/21   0.52381
G        4/7    4/7     3/7  11/21   0.52381
H4       6/7    6/7     5/7  17/21  0.809524
H4u3     6/7    6/7     5/7  17/21  0.809524
H4fix    5/7    6/7     5/7  16/21  0.761905

Agreement with ATLAS, 2019->2023 (sensitivity):
      C1_dir C2_abs C3_rank  total     share
NB       5/7    5/7     2/7  12/21  0.571429
G        1/7    2/7     3/7   6/21  0.285714
H4       6/7    7/7     5/7  18/21  0.857143
H4u3     6/7    7/7     5/7  18/21  0.857143
H4fix    6/7    6/7     6/7  18/21  0.857143



Takeaway: this is the primary metric. The text cell after the Summary gives the reading, with numbers taken from this output.

In [9]:
def n_changed(df, a, b):
    out = {}
    for col in ['C1_dir', 'C2_abs', 'C2_pct', 'C3_rank']:
        x, y = wide(df, col)[a], wide(df, col)[b]
        ok = x.notna() & y.notna()
        out[col] = f'{int((x[ok] != y[ok]).sum())}/{int(ok.sum())}'
        out[col + '_which'] = ', '.join(x[ok][x[ok] != y[ok]].index)
    return out
chg = pd.DataFrame({f'{a}->{b}': n_changed(prim, a, b) for a, b in [('NB', 'H4'), ('G', 'H4'), ('NB', 'G'), ('H4', 'H4u3'), ('H4', 'H4fix')]}).T
chg

,C1_dir,C1_dir_which,C2_abs,C2_abs_which,C2_pct,C2_pct_which,C3_rank,C3_rank_which
NB->H4,3/7,"CAN, ARE, GBR",3/7,"POL, ARE, VNM",5/7,"DEU, POL, SGP, GBR, VNM",6/7,"CAN, DEU, POL, ARE, GBR, VNM"
G->H4,3/7,"CAN, SGP, ARE",1/7,ARE,2/7,"CAN, ARE",3/7,"SGP, ARE, VNM"
NB->G,2/7,"SGP, GBR",2/7,"POL, VNM",7/7,"CAN, DEU, POL, SGP, ARE, GBR, VNM",6/7,"CAN, DEU, POL, SGP, GBR, VNM"
H4->H4u3,0/7,,0/7,,0/7,,0/7,
H4->H4fix,3/7,"CAN, SGP, VNM",3/7,"CAN, ARE, VNM",1/7,VNM,0/7,


Takeaway: this shows how many report claims change from the current method (`NB`) to the HS4 replacement, and how much of that change is just the sign fix (`NB→G`) versus the move to products (`G→H4`).

In [10]:
d1 = wide(prim, 'C1_dir')
rob = d1[['H4', 'H4u3', 'H4fix', 'ATLAS']].nunique(axis=1).eq(1) & d1[['H4', 'H4u3', 'H4fix', 'ATLAS']].notna().all(axis=1)
chgH = prim[prim.variant == 'H4'].set_index('country')['change']
chgA = prim[prim.variant == 'ATLAS'].set_index('country')['change']
robtab = pd.DataFrame({'H4_change': chgH, 'ATLAS_change': chgA, 'H4_dir': d1['H4'], 'ATLAS_dir': d1['ATLAS'], 'robust_dir': rob}).reindex(REPORT)
print('robust C1 calls (2019->2022):', int(rob.sum()), 'of', len(rob)); robtab.round(3)

robust C1 calls (2019->2022): 4 of 7


,H4_change,ATLAS_change,H4_dir,ATLAS_dir,robust_dir
country,,,,,
CAN,-0.022,-0.049,decreased,decreased,False
DEU,-0.125,-0.024,decreased,decreased,True
POL,-0.207,-0.030,decreased,decreased,True
SGP,-0.038,-0.031,decreased,decreased,False
ARE,-0.236,-0.015,decreased,decreased,True
GBR,0.107,0.088,increased,increased,True
VNM,-0.072,0.074,decreased,increased,False


Takeaway: a direction word is only worth printing if it is robust. Where the change is a few hundredths of an SD, the word is noise.

In [11]:
# How big are these changes relative to ordinary year-to-year movement? (H4, all countries in the fixed panel)
yy = pd.concat([(V['H4fix'][y + 1] - V['H4fix'][y]).abs() for y in [2019, 2020, 2021]])
print('H4fix |one-year change|, all countries 2019-2022: median %.3f, 75th pct %.3f, 90th pct %.3f' % (yy.median(), yy.quantile(.75), yy.quantile(.9)))
three = (V['H4fix'][2022] - V['H4fix'][2019]).abs()
print('H4fix |2019->2022 change|, all countries: median %.3f' % three.median())
print('report countries |H4 2019->2022 change|:'); print(chgH.abs().reindex(REPORT).round(3).to_string())

H4fix |one-year change|, all countries 2019-2022: median 0.071, 75th pct 0.141, 90th pct 0.269
H4fix |2019->2022 change|, all countries: median 0.107
report countries |H4 2019->2022 change|:
country
CAN    0.022
DEU    0.125
POL    0.207
SGP    0.038
ARE    0.236
GBR    0.107
VNM    0.072


Takeaway: this puts the report countries' changes next to the typical change. A change below the all-country median is not a story.

### E. The % change formula

In [12]:
pc = wide(prim, 'pct_change').round(1)
st = wide(prim, 'start').round(3)
pd.concat({'start_ECI': st[['NB', 'G', 'H4', 'ATLAS']], 'pct_change': pc[['NB', 'G', 'H4', 'ATLAS']]}, axis=1)

start_ECI                      pct_change                    
variant        NB      G     H4  ATLAS         NB      G     H4 ATLAS
country                                                              
CAN        -0.853 -1.058  0.462  0.916      175.4   54.0   -4.7  -5.3
DEU         1.988  2.031  1.843  1.544     -180.1  -19.1   -6.8  -1.6
POL         0.951  0.598  0.827  1.060     -165.4  -38.4  -25.0  -2.8
SGP         0.764  0.843  2.108  1.684     -298.7   94.5   -1.8  -1.8
ARE        -2.079 -2.051  0.149  0.701      195.8    9.4 -158.3  -2.1
GBR         0.509  0.461  1.439  1.309     -394.4  223.2    7.4   6.7
VNM         0.741  0.690  0.040  0.403     -168.5  -30.8 -178.7  18.4

Takeaway: dividing by |start| blows up whenever the start value is near 0. That is common because ECI is standardized to mean 0. The % figure is therefore not comparable across countries, and the C2 calls in the `C2_pct` column can flip for that reason alone.

### F. Presentation: does HS4 PCI aggregated to the 18 groups agree with the grouped PCI? (2019)

In [13]:
y = 2019
X, M = HS4X[y], HS4M[y]
pci4 = eci.eci_pci(M)[1]
w = X[pci4.index].sum(axis=0)
rows = {}
for gname, chs in R.LAB_GROUPS.items():
    prods = [p for p in pci4.index if int(p[:2]) in chs]
    rows[gname] = dict(n_hs4=len(prods), hs4_pci_wmean=float(np.average(pci4[prods], weights=w[prods])) if prods else np.nan,
                       grouped_pci=float(PCI_G[y].get(gname.strip(), np.nan)) if gname.strip() in PCI_G[y].index else np.nan)
gp = pd.DataFrame(rows).T
gp.index = gp.index.str.strip()
gp['grouped_pci'] = [PCI_G[y].get(i, np.nan) for i in gp.index]
gp['rank_hs4'] = gp.hs4_pci_wmean.rank(ascending=False); gp['rank_grouped'] = gp.grouped_pci.rank(ascending=False)
print('Spearman(group PCI: HS4-aggregated vs grouped), 2019:', round(R.spearman(gp.hs4_pci_wmean, gp.grouped_pci)[0], 3), ' n =', R.spearman(gp.hs4_pci_wmean, gp.grouped_pci)[1])
gp.sort_values('rank_hs4').round(3)

Spearman(group PCI: HS4-aggregated vs grouped), 2019: 0.922  n = 17


,n_hs4,hs4_pci_wmean,grouped_pci,rank_hs4,rank_grouped
High-complexity Products,244.0,0.945,1.716,1.0,2.0
Transportation,76.0,0.937,1.872,2.0,1.0
Daily Instruments,73.0,0.737,1.065,3.0,3.0
Chemicals,92.0,0.706,0.526,4.0,6.0
Chemical Manufacturing,73.0,0.506,0.356,5.0,7.0
Machinery and Construction,11.0,0.494,1.055,6.0,4.0
Weapons,13.0,0.448,0.165,7.0,9.0
Raw Manufacturing,101.0,0.315,0.649,8.0,5.0
Metals,137.0,0.112,-0.330,9.0,12.0
Wood Products,65.0,-0.264,-0.302,10.0,11.0


Takeaway: this shows whether the industry-complexity ordering in Table 1 / §4 would survive the switch. If it doesn't, the report's "most complex industries" wording has to change too.

### G. Robustness of the primary metric to the window

In [14]:
summ = pd.DataFrame({f'{w[0]}->{w[1]}': AG[w]['share'] for w in AG}).round(3)
summ

,2019->2022,2019->2021,2019->2023
NB,0.333333,0.52381,0.571429
G,0.52381,0.52381,0.285714
H4,0.761905,0.809524,0.857143
H4u3,0.761905,0.809524,0.857143
H4fix,0.761905,0.761905,0.857143


Takeaway: this is the primary metric for all three windows. 2019→2023 inherits the D1 problems.

## Summary of results (numbers pulled from the outputs above)

In [15]:
A = AG[(2019, 2022)]
print('PRIMARY (2019->2022) share of 21 claims agreeing with ATLAS:')
for k in A.index: print(f'  {k:6s} {A.loc[k, "total"]:>6s}  = {A.loc[k, "share"]:.3f}   [C1 {A.loc[k,"C1_dir"]}, C2 {A.loc[k,"C2_abs"]}, C3 {A.loc[k,"C3_rank"]}]')
print('NB->H4 claims changed:', chg.loc['NB->H4', ['C1_dir', 'C2_abs', 'C2_pct', 'C3_rank']].to_dict())
print('G->H4  claims changed:', chg.loc['G->H4', ['C1_dir', 'C2_abs', 'C2_pct', 'C3_rank']].to_dict())
print('H4->H4u3 claims changed:', chg.loc['H4->H4u3', ['C1_dir', 'C2_abs', 'C3_rank']].to_dict())
print('H4->H4fix claims changed:', chg.loc['H4->H4fix', ['C1_dir', 'C2_abs', 'C3_rank']].to_dict())
print('robust direction calls:', int(rob.sum()), 'of 7:', ', '.join(rob[rob].index))
# max rank change at HS4 from the u>=3 floor, report countries + peers, every year (proposal 2026-11-13 #2)
mx = {}
for y in YEARS:
    r1 = V['H4'][y].rank(ascending=False); r2 = V['H4u3'][y].rank(ascending=False)
    f = [c for c in FOCUS if c in r1.index]
    mx[y] = float((r1[f] - r2[f]).abs().max())
print('max |rank change| H4 -> H4u3 among report countries+peers by year:', mx)
print('runtime: %.1f s' % (time.time() - T0))

PRIMARY (2019->2022) share of 21 claims agreeing with ATLAS:
  NB       7/21  = 0.333   [C1 3/7, C2 4/7, C3 0/7]
  G       11/21  = 0.524   [C1 3/7, C2 4/7, C3 4/7]
  H4      16/21  = 0.762   [C1 6/7, C2 5/7, C3 5/7]
  H4u3    16/21  = 0.762   [C1 6/7, C2 5/7, C3 5/7]
  H4fix   16/21  = 0.762   [C1 5/7, C2 6/7, C3 5/7]
NB->H4 claims changed: {'C1_dir': '3/7', 'C2_abs': '3/7', 'C2_pct': '5/7', 'C3_rank': '6/7'}
G->H4  claims changed: {'C1_dir': '3/7', 'C2_abs': '1/7', 'C2_pct': '2/7', 'C3_rank': '3/7'}
H4->H4u3 claims changed: {'C1_dir': '0/7', 'C2_abs': '0/7', 'C3_rank': '0/7'}
H4->H4fix claims changed: {'C1_dir': '3/7', 'C2_abs': '3/7', 'C3_rank': '0/7'}
robust direction calls: 4 of 7: DEU, POL, ARE, GBR
max |rank change| H4 -> H4u3 among report countries+peers by year: {2019: 1.0, 2020: 1.0, 2021: 0.0, 2022: 0.0, 2023: 1.0}
runtime: 3.5 s


## Interpretation

**Answer:** yes. Porting the reports to oriented HS4 ECI is worth doing, and it changes what most reports say. On 2019 → 2022, HS4 claims agree with ATLAS on 16/21, against 7/21 for the current method. About half of the gain comes from fixing the sign (`NB → G`: 7 → 11), and the rest comes from moving to products (`G → H4`: 11 → 16).

But the port alone isn't enough. Three more changes would make the §2.2 paragraph honest:

1. **Compute changes on a fixed country panel** (`H4fix`). On the primary window it ties H4 on the total (16/21) and is better on peer standing (6/7 vs 5/7). On 2019 → 2023 it is the only variant at 6/7 on every claim type. It also removes the sample-churn flips (Canada, Singapore, Vietnam in H4 → H4fix).
2. **Print the absolute change in SD units and drop the % change.** The % change mixes up "small change from near zero" and "large change", so the C2 calls built on it disagree with the absolute version for several countries (Section C, `C2_pct` vs `C2_abs`).
3. **Call a change "essentially unchanged" when it is smaller than ordinary year-to-year movement.** On this panel the median one-year |Δ| is 0.071 SD. A threshold like that would have stopped the Canada and Singapore reports from printing a direction at all. The threshold itself is a proposal and needs calibrating (avenue 1).

**What still disagrees with ATLAS under HS4 (2019 → 2022):** Vietnam's direction and peer standing, Canada's peer standing, and Singapore's and the UAE's level rank among peers. The UAE's rank partly reflects coverage: Saudi Arabia is missing from `2022_mega.csv`, so the UAE is ranked out of 3 rather than 4. Its HS4 ECI (−0.087 in 2022) also sits far below ATLAS (0.686), possibly because of exporter-reported oil and re-exports (untested). Singapore ranks 2/4 under HS4 but 3/4 under ATLAS. Vietnam's and Canada's changes are small (Section D), so their calls are fragile under any variant. Whether the rest come from data or method is avenue 3.

**Section F (presentation):** if HS4 PCI is averaged to the lab's groups, the top three stay the same (High-complexity, Transportation, Daily Instruments). Clothing falls from 10th to 14th, Raw Manufacturing from 5th to 8th, and Metals rises from 12th to 9th. The groups' wording would need light edits, not a rewrite. Services and Utilities has no HS4 products (chapter 99 is dropped), so a services statement would need its own source.

**Not shown here:** nothing about growth or causality. This is purely about whether the reports describe the measured ECI consistently with the reference implementation.

## Caveats

- **Seven countries and three peers each.** The 21-claim agreement share is a small count. One or two claims either way change the share by 5–10 points, so read it as directional.
- **ATLAS is a benchmark, not ground truth.** It uses mirror-reconciled HS92 trade, a population and export-size filter, and its own cleaning. Its year-to-year changes are also small and noisy. Agreeing with it means "says what the Growth Lab would say", not "is right".
- **The window is short and not the reports' window.** The reports say 2019 → 2024. There is no 2024 data in the repo, so these are dry runs on 2019 → 2022, with 2019 → 2021 and 2019 → 2023 as checks. The printed Spring 2026 values still can't be reproduced (data request 2026-11-11).
- **Missing countries.** Saudi Arabia has no exports in `2022_mega.csv` (data request 2026-11-12-2023-repull), so it drops out of the UAE peer set under every repo-data variant in 2022. Grouped variants also lose Qatar, Kuwait and the Philippines in some years because of the drop-any-zero filter (D2). Peer averages therefore rest on 1–3 peers depending on the variant (see `peers_used`).
- **Nominal values, exporter-reported data, HS revisions mixed across reporters** (data request 2026-11-13). 2023 is suspect (D1). Grouped variants inherit the D4 mapping problems and the EU-28 row (Q02).
- `C3` compares the rank only. When a peer is missing in one variant, the group size differs.
- In the Section B table, the UK's grouped ECI (`G`) jumps to −2.72 in 2023, while H4 and ATLAS stay flat. That's another sign that the 2023 grouped file is off (D1). It is why `G` drops to 6/21 on the 2019 → 2023 window.

## Helpful to know

- `NB` has to be computed on `eci.load_groups(notebook_names=False)` in the **original CSV row order**. The notebook's sign depends on row 0, so re-sorting by ISO3 first would silently change `NB`.
- `research.grouped_eci` keeps the `EU28` row (as the notebook does). `research.repo_hs` drops it.
- At HS4 the ubiquity floor u ≥ 3 is nearly a no-op (Section A level fit and the max-rank-change line in the Summary). It's a cheap guard to keep if the reports move to HS4, but it isn't needed the way it is at HS6 (P-2026-11-13-1).
- The fixed HS4 panel (countries with ≥ $1B in every year 2019–2023) is printed in Section A. It's the right base for any "ECI changed from X to Y" sentence, because the per-year sample churns.

## New research avenues

Each is scoped to one night (also in `proposals/2026-11-14.md`).

1. **Calibrate an "essentially unchanged" band for report sentences.** Use Q09's product bootstrap on the fixed HS4 panel to estimate, for each Spring 2026 country, the probability that the 2019 → 2022 change has the printed sign. Propose a threshold (in SD) below which the template prints "essentially unchanged".
2. **Draft the HS4 version of the §2.2 JSON fields.** Write a small function, kept out of `code/current`, that emits `[ECI start]`, `[ECI end]`, `[ECI change]` (SD units), `[increased/decreased/essentially unchanged]`, and `[ECI peer standing]` from fixed-panel HS4 ECI. Diff its output against tonight's table for the 7 countries.
3. **Separate "our trade data" from "our method" in the remaining 5 disagreements.** Run the same claims on the Growth Lab HS4 export panel (`external-data/raw/atlas_hs4_exports.pkl`, gitignored, rebuilt with `download.sh`) through `lib/eci.py`. Where that agrees with ATLAS but repo-HS4 doesn't, the cause is the data (UAE, Saudi Arabia).
4. **Widen the dry run to the Spring 2025 and Fall 2025 report countries** (peers from `catalog/COUNTRIES.md` and the PDFs). This roughly triples the number of claims and tightens the agreement shares.
5. **Services gap.** HS4 drops chapter 99, so Services and Utilities can't be scored. Measure how much the grouped Services column drives Singapore's, the UK's and the UAE's grouped ECI, to decide whether HS4 reports need a separate services note.

## Data issues / data needed

No new data request tonight. Existing ones that touch this result:
- `data-requests/2026-11-11-report-eci-provenance.md`: the Spring 2026 printed ECIs still can't be reproduced, and there is no 2024 file.
- `data-requests/2026-11-12-2023-repull.md`: Saudi Arabia is missing from `2022_mega.csv`, which removes it from the UAE peer set in 2022.